# 16 — UCI Multicenter Data Processing

Lab này đo tác động của preprocessing trên cùng bộ 920 bệnh nhân thật và cùng thiết kế LOCO của Lab 15. Logistic Regression và LightGBM chỉ đóng vai trò downstream evaluator.

Không dùng synthetic data, SMOTE hoặc threshold tuning trong lab này. Mục tiêu là tách riêng ảnh hưởng của sentinel handling, missingness và outlier processing.

> Đây là nghiên cứu trên dữ liệu công khai, không phải bằng chứng lâm sàng hay công cụ chẩn đoán.

In [ ]:
!pip -q install lightgbm seaborn

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, brier_score_loss, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
THRESHOLD = 0.50
OUTPUT_DIR = Path('/content/uci_multicenter_data_processing_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 100)
print('Output directory:', OUTPUT_DIR)

## 1. Tải 920 dòng thật và giữ nguyên dữ liệu raw

Các quy tắc xử lý chỉ được áp dụng trên bản copy của training/test fold. Không impute hoặc clip toàn bộ 920 dòng trước khi chia LOCO.

In [ ]:
COLUMNS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'
]
FEATURES = COLUMNS[:-1]
NUMERICAL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {
    'cleveland': 'processed.cleveland.data',
    'hungarian': 'processed.hungarian.data',
    'switzerland': 'processed.switzerland.data',
    'va': 'processed.va.data',
}
EXPECTED_ROWS = {'cleveland': 303, 'hungarian': 294, 'switzerland': 123, 'va': 200}

frames = []
for site, filename in FILES.items():
    url = f'{BASE_URL}/{filename}'
    part = pd.read_csv(url, names=COLUMNS, na_values=['?'], skipinitialspace=True)
    part = part.apply(pd.to_numeric, errors='coerce')
    part['site'] = site
    part['target'] = (part['num'] > 0).astype('int8')
    assert len(part) == EXPECTED_ROWS[site], f'{site}: expected {EXPECTED_ROWS[site]}, got {len(part)}'
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
assert len(df) == 920, f'Expected 920 rows, got {len(df)}'
df[FEATURES + ['num', 'target', 'site']].to_csv(OUTPUT_DIR / 'uci_multicenter_raw.csv', index=False)
print('Combined shape:', df.shape)
display(df.head())

In [ ]:
site_summary = df.groupby('site').agg(
    rows=('target', 'size'),
    disease_count=('target', 'sum'),
    positive_rate=('target', 'mean'),
).reset_index()
missing_by_site = df.groupby('site')[FEATURES].apply(lambda x: x.isna().mean()).T
site_summary['missing_cells'] = site_summary['site'].map(
    df.groupby('site')[FEATURES].apply(lambda x: int(x.isna().sum().sum()))
)
site_summary['missing_rate'] = site_summary['missing_cells'] / (site_summary['rows'] * len(FEATURES))

display(site_summary.round(4))
display((missing_by_site * 100).round(1))
print('Duplicate rows excluding site:', int(df[COLUMNS].duplicated().sum()))
site_summary.to_csv(OUTPUT_DIR / 'site_summary.csv', index=False)
missing_by_site.to_csv(OUTPUT_DIR / 'missing_by_site.csv')

## 2. Bốn processing configurations

- **P0_baseline:** giữ raw values; imputation trong pipeline.
- **P1_sentinel_aware:** đổi `trestbps <= 0` và `chol <= 0` thành missing theo quy tắc định trước.
- **P2_sentinel_outlier_flags:** P1 + thêm cờ outlier học từ training fold.
- **P3_sentinel_outlier_clip_robust:** P2 + clip numerical theo IQR của training fold + RobustScaler.

Không xóa bệnh nhân chỉ vì outlier. Cờ outlier được thêm để model có thể học sự bất thường; clipping chỉ là một nhánh sensitivity analysis.

In [ ]:
PROCESSING_CONFIGS = {
    'P0_baseline': {'sentinel_aware': False, 'outlier_flags': False, 'clip_outliers': False, 'scaler': 'standard'},
    'P1_sentinel_aware': {'sentinel_aware': True, 'outlier_flags': False, 'clip_outliers': False, 'scaler': 'standard'},
    'P2_sentinel_outlier_flags': {'sentinel_aware': True, 'outlier_flags': True, 'clip_outliers': False, 'scaler': 'standard'},
    'P3_sentinel_outlier_clip_robust': {'sentinel_aware': True, 'outlier_flags': True, 'clip_outliers': True, 'scaler': 'robust'},
}

def replace_sentinels(frame, enabled):
    out = frame.copy()
    counts = {}
    for col in ['trestbps', 'chol']:
        mask = out[col].notna() & (out[col] <= 0)
        counts[col] = int(mask.sum()) if enabled else 0
        if enabled:
            out.loc[mask, col] = np.nan
    return out, counts

def fit_iqr_bounds(train_frame):
    bounds = {}
    for col in NUMERICAL_FEATURES:
        values = train_frame[col].dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds

def add_outlier_flags(frame, bounds, clip_values=False):
    out = frame.copy()
    for col, (lower, upper) in bounds.items():
        flag_name = f'{col}__outlier'
        mask = out[col].notna() & ((out[col] < lower) | (out[col] > upper))
        out[flag_name] = mask.astype('int8')
        if clip_values:
            out[col] = out[col].clip(lower=lower, upper=upper)
    return out

def prepare_fold_data(X_train, X_test, config):
    train, train_sentinel_counts = replace_sentinels(X_train, config['sentinel_aware'])
    test, test_sentinel_counts = replace_sentinels(X_test, config['sentinel_aware'])
    numeric_features = list(NUMERICAL_FEATURES)
    audit = {
        'sentinel_train_total': int(sum(train_sentinel_counts.values())),
        'sentinel_test_total': int(sum(test_sentinel_counts.values())),
        'outlier_train_total': 0,
        'outlier_test_total': 0,
        'clipping_enabled': bool(config['clip_outliers']),
    }
    if config['outlier_flags']:
        bounds = fit_iqr_bounds(train)
        train = add_outlier_flags(train, bounds, config['clip_outliers'])
        test = add_outlier_flags(test, bounds, config['clip_outliers'])
        flag_features = [f'{col}__outlier' for col in NUMERICAL_FEATURES]
        numeric_features.extend(flag_features)
        audit['outlier_train_total'] = int(train[flag_features].sum().sum())
        audit['outlier_test_total'] = int(test[flag_features].sum().sum())
    return train, test, numeric_features, audit

In [ ]:
def make_preprocessor(numeric_features, scaler_name):
    scaler = StandardScaler() if scaler_name == 'standard' else RobustScaler()
    numeric = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', scaler),
    ])
    categorical = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ])
    return ColumnTransformer([
        ('numerical', numeric, numeric_features),
        ('categorical', categorical, CATEGORICAL_FEATURES),
    ])

def make_models():
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=250, learning_rate=0.03, num_leaves=15,
            min_child_samples=15, subsample=0.9, colsample_bytree=0.9,
            reg_lambda=1.0, class_weight='balanced',
            random_state=RANDOM_STATE, verbosity=-1
        ),
    }

def metric_record(processing, model_name, test_site, y_true, probability, fit_seconds):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        'processing': processing, 'model': model_name, 'test_site': test_site,
        'test_rows': int(len(y_true)),
        'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability) if len(np.unique(y_true)) == 2 else np.nan,
        'brier': brier_score_loss(y_true, probability),
        'false_negatives': int(fn), 'fit_seconds': fit_seconds,
    }

## 3. LOCO evaluation cho từng processing configuration

Mỗi processing configuration được fit độc lập trong từng outer fold. IQR bounds, sentinel transformation, imputer và scaler không nhìn thấy test hospital.

In [ ]:
records = []
processing_audits = []

for processing_name, config in PROCESSING_CONFIGS.items():
    for test_site in FILES:
        train_df = df[df['site'] != test_site].copy()
        test_df = df[df['site'] == test_site].copy()
        X_train_raw, y_train = train_df[FEATURES], train_df['target']
        X_test_raw, y_test = test_df[FEATURES], test_df['target']

        X_train, X_test, numeric_features, audit = prepare_fold_data(
            X_train_raw, X_test_raw, config
        )
        processing_audits.append({
            'processing': processing_name,
            'test_site': test_site,
            'train_rows': len(X_train),
            'test_rows': len(X_test),
            **audit,
        })

        for model_name, classifier in make_models().items():
            pipeline = Pipeline([
                ('preprocessor', make_preprocessor(numeric_features, config['scaler'])),
                ('classifier', classifier),
            ])
            started = time.perf_counter()
            pipeline.fit(X_train, y_train)
            fit_seconds = time.perf_counter() - started
            probability = pipeline.predict_proba(X_test)[:, 1]
            records.append(metric_record(
                processing_name, model_name, test_site, y_test, probability, fit_seconds
            ))

processing_results = pd.DataFrame(records).sort_values(
    ['processing', 'model', 'test_site']
).reset_index(drop=True)
processing_audit = pd.DataFrame(processing_audits)
processing_results.to_csv(OUTPUT_DIR / 'data_processing_loco_results.csv', index=False)
processing_audit.to_csv(OUTPUT_DIR / 'data_processing_fold_audit.csv', index=False)
display(processing_results.round(4))

In [ ]:
summary = processing_results.groupby(['processing', 'model']).agg(
    roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'),
    roc_auc_worst=('roc_auc', 'min'),
    recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'),
    recall_worst=('recall', 'min'),
    specificity_mean=('specificity', 'mean'),
    f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'),
    false_negatives_total=('false_negatives', 'sum'),
    fit_seconds_mean=('fit_seconds', 'mean'),
).reset_index().sort_values(
    ['model', 'roc_auc_worst', 'recall_worst'], ascending=[True, False, False]
)

print('DATA PROCESSING SUMMARY')
display(summary.round(6))
summary.round(6).to_csv(OUTPUT_DIR / 'data_processing_summary.csv', index=False)
summary.round(6).to_json(OUTPUT_DIR / 'data_processing_summary.json', orient='records', indent=2)

baseline = summary[summary['processing'] == 'P0_baseline'].set_index('model')
delta = summary.set_index(['processing', 'model']).copy()
for metric in ['roc_auc_mean', 'roc_auc_worst', 'recall_mean', 'recall_worst', 'brier_mean', 'false_negatives_total']:
    delta[f'delta_vs_P0_{metric}'] = [
        row[metric] - baseline.loc[model, metric]
        for (processing, model), row in delta.iterrows()
    ]
delta.reset_index().round(6).to_csv(OUTPUT_DIR / 'data_processing_delta_vs_baseline.csv', index=False)
display(delta.reset_index().round(6))

In [ ]:
plot_data = summary.pivot(index='processing', columns='model', values='roc_auc_worst')
ax = plot_data.plot(kind='bar', figsize=(10, 4), ylim=(0.5, 1.0), rot=20)
ax.set_ylabel('Worst-site ROC-AUC')
ax.set_title('Data processing impact on worst-site external performance')
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'processing_worst_site_auc.png', dpi=180, bbox_inches='tight')
plt.show()

heat = summary.pivot(index='processing', columns='model', values='brier_mean')
plt.figure(figsize=(7, 4))
sns.heatmap(heat, annot=True, fmt='.4f', cmap='Blues_r')
plt.title('Mean Brier score by processing and model')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'processing_brier_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. Cách đọc kết quả

- Không chọn processing chỉ vì ROC-AUC trung bình tăng.
- Ưu tiên worst-site ROC-AUC, sau đó worst-site Recall, Brier và tổng false negatives.
- Nếu P1/P2/P3 không cải thiện ổn định trên cả hai model, giữ P0 làm baseline và không tuyên bố preprocessing mới tốt hơn.
- Kết quả của lab này chưa bao gồm class imbalance intervention hoặc synthetic augmentation; hai phần đó phải là các ablation riêng.

In [ ]:
run_config = {
    'dataset_rows': 920,
    'validation': 'Leave-One-Center-Out',
    'threshold': THRESHOLD,
    'random_state': RANDOM_STATE,
    'synthetic_data': False,
    'smote': False,
    'processing_configs': PROCESSING_CONFIGS,
    'sentinel_rule': {'trestbps': '<= 0 -> NaN', 'chol': '<= 0 -> NaN'},
    'source_urls': {site: f'{BASE_URL}/{filename}' for site, filename in FILES.items()},
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/content/uci_multicenter_data_processing_results', 'zip', OUTPUT_DIR)
print('Saved artifacts to:', OUTPUT_DIR)
print('ZIP:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Không chạy trong Colab; ZIP vẫn nằm tại:', zip_path)

## Checklist

- [x] 920 bệnh nhân thật từ 4 cohort.
- [x] LOCO giữ nguyên như Lab 15.
- [x] Sentinel và IQR bounds chỉ fit trên training fold.
- [x] Imputation và scaling nằm trong Pipeline.
- [x] Không xóa bệnh nhân vì outlier.
- [x] Không synthetic, không SMOTE, không threshold tuning.
- [ ] Chọn processing configuration sau khi xem kết quả từng site.